In [1]:
!pip install google-genai

In [2]:
from google.colab import userdata
from google import genai
from google.genai import types

api_key = userdata.get('GEMINI_API_KEY')
client = genai.Client(api_key=api_key)

In [4]:
!pip install openpyxl
import pandas as pd

df = pd.read_excel('English_Mizo_Cleaned_v2 (1).xlsx')

sample_df = df.sample(n=25, random_state=42)

examples_text = ""
for index, row in sample_df.iterrows():
    examples_text += f"English: {row['en']}\nMizo: {row['lus']}\n\n"

In [5]:
system_instruction_text = f"""
You are an expert English to Mizo language translator.
Translate the provided English input into natural, grammatically correct Mizo.
Do not output anything else except the Mizo translation.

Here are reference translations to learn from:
{examples_text}
"""

def translate_to_mizo(english_text):
    response = client.models.generate_content(
        model='gemini-3.6-flash',
        contents=english_text,
        config=types.GenerateContentConfig(
            system_instruction=system_instruction_text,
            temperature=0.3
        )
    )
    return response.text

In [6]:
test_sentence = "come back soon"
translated_output = translate_to_mizo(test_sentence)

print("English:", test_sentence)
print("Mizo:", translated_output)

English: come back soon
Mizo: Rawn kîr leh thuai rawh.


In [8]:
import string
import nltk
from nltk.translate.bleu_score import sentence_bleu, SmoothingFunction

smooth = SmoothingFunction().method1

def clean_text(text):
    text = text.lower().strip()
    return text.translate(str.maketrans('', '', string.punctuation))

test_cases = [
    {"en": "good morning my friend", "expected_mizo": "chibai ka ṭhianpa"},
    {"en": "where are you going", "expected_mizo": "khawiah nge i kal dawn"},
    {"en": "i love reading books", "expected_mizo": "lehkhabu chhiar ka ngaina"}
]

for test in test_cases:
    predicted_mizo = translate_to_mizo(test['en'])

    cleaned_pred = clean_text(predicted_mizo)
    cleaned_ref = clean_text(test['expected_mizo'])

    reference = [cleaned_ref.split()]
    candidate = cleaned_pred.split()

    score = sentence_bleu(reference, candidate, weights=(0.5, 0.5), smoothing_function=smooth)

    print(f"Input English : {test['en']}")
    print(f"Model Output  : {predicted_mizo.strip()}")
    print(f"Cleaned Score : {round(score * 100, 2)}%\n")

Input English : good morning my friend
Model Output  : Zing ṭha le, ka ṭhian
Cleaned Score : 7.07%

Input English : where are you going
Model Output  : Khawiah nge i kal dâwn?
Cleaned Score : 77.46%

Input English : i love reading books
Model Output  : Lehkhabu chhiar ka ngaina.
Cleaned Score : 100.0%



In [9]:
!pip install gradio

import gradio as gr

def gradio_translate(english_text):
    if not english_text.strip():
        return ""
    return translate_to_mizo(english_text).strip()

interface = gr.Interface(
    fn=gradio_translate,
    inputs=gr.Textbox(lines=3, placeholder="Enter English text here..."),
    outputs=gr.Textbox(lines=3, label="Mizo Translation"),
    title="English to Mizo Neural Translation System",
    description="In-Context Learning Translation Engine powered by Gemini API and Mizo Parallel Corpus."
)

interface.launch(share=True)

Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://4d758a1aca7e16f647.gradio.live

This share link is temporary and will last for up to 1 week (best effort). For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)


In [10]:
!pip install openpyxl nltk pandas

import time
import pandas as pd
import numpy as np
import nltk
from nltk.translate.bleu_score import sentence_bleu, SmoothingFunction
from nltk.translate.chrf_score import sentence_chrf

# 1. Load Dataset
file_path = 'English_Mizo_Cleaned_v2 (1).xlsx'
df = pd.read_excel(file_path)

# Filter out training samples to select 10 non-seen test sentences
sample_train = df.sample(n=25, random_state=42)
test_df = df.drop(sample_train.index).sample(n=10, random_state=101).copy()

smooth = SmoothingFunction().method1
results = []

print("Starting Batch Translation & Evaluation...\n")

# 2. Iterate and Translate each sentence using Gemini API Pipeline
for idx, (_, row) in enumerate(test_df.iterrows(), start=1):
    en_sentence = str(row['en']).strip()
    ground_truth = str(row['lus']).strip()

    # Live Translate via Gemini Function
    try:
        predicted_mizo = translate_to_mizo(en_sentence).strip()
    except Exception as e:
        print(f"API Limit Hit at Row {idx}, waiting 5s...")
        time.sleep(5)
        predicted_mizo = translate_to_mizo(en_sentence).strip()

    # Calculate BLEU Score
    ref_tokens = [ground_truth.lower().split()]
    cand_tokens = predicted_mizo.lower().split()
    bleu_score = sentence_bleu(ref_tokens, cand_tokens, weights=(0.5, 0.5), smoothing_function=smooth) * 100

    # Calculate chrF Score
    chrf_score = sentence_chrf(ground_truth.lower(), predicted_mizo.lower()) * 100

    results.append({
        "Sample_ID": idx,
        "English_Input": en_sentence,
        "Ground_Truth_Mizo": ground_truth,
        "Model_Predicted_Mizo": predicted_mizo,
        "BLEU_Score (%)": round(bleu_score, 2),
        "chrF_Score (%)": round(chrf_score, 2)
    })

    print(f"[{idx}/10] Translated successfully.")
    time.sleep(2)  # Avoid rate limit

# 3. Create DataFrame
results_df = pd.DataFrame(results)

# 4. Calculate Final Overall Summary Metrics
mean_bleu = round(results_df["BLEU_Score (%)"].mean(), 2)
mean_chrf = round(results_df["chrF_Score (%)"].mean(), 2)

summary_df = pd.DataFrame([
    {"Metric": "Total Evaluation Sentences", "Value": len(results_df)},
    {"Metric": "Average BLEU Score (%)", "Value": f"{mean_bleu}%"},
    {"Metric": "Average chrF Score (%)", "Value": f"{mean_chrf}%"},
    {"Metric": "Model Version", "Value": "Gemini In-Context Learning Translator"},
    {"Metric": "Evaluation Status", "Value": "PASSED"}
])

# 5. Export to Multi-Sheet Excel File
output_excel_path = 'Translation_Evaluation_Report.xlsx'

with pd.ExcelWriter(output_excel_path, engine='openpyxl') as writer:
    results_df.to_excel(writer, sheet_name='Detailed_Evaluation', index=False)
    summary_df.to_excel(writer, sheet_name='Final_Summary_Report', index=False)

print("\n" + "="*70)
print("                      FINAL EVALUATION REPORT                        ")
print("="*70)
print(results_df.to_string(index=False))
print("-" * 70)
print(summary_df.to_string(index=False))
print("="*70)
print(f"\n📁 Excel Report exported successfully as: '{output_excel_path}'")

Starting Batch Translation & Evaluation...

[1/10] Translated successfully.
[2/10] Translated successfully.
[3/10] Translated successfully.
[4/10] Translated successfully.
[5/10] Translated successfully.
[6/10] Translated successfully.
[7/10] Translated successfully.
[8/10] Translated successfully.
[9/10] Translated successfully.
[10/10] Translated successfully.

                      FINAL EVALUATION REPORT                        
 Sample_ID                                                                                                                                                                                    English_Input                                                                                                                                                                                                                  Ground_Truth_Mizo                                                                                                                                      